This program cleans the enron email files and creates a .CSV file of X amount of prepared emails to be used in ML models

Run on python 3.13.13 kernel, on Visual studios code, using juypter notebook
Generates "1_Enron_ml_ready.csv" which contains a label column and a text column, comprising of email subject + email body.
Runs on the Enron email corpus.

In [1]:
%pip install pandas scikit-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\jwboy\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Starting with a small sample batch to prove extraction and
text cleaning works.
If successful, will then be scaled across the desired amount of emails needed for the dataset 

In [2]:
#checking that the desired folder exists
import os #allows for interacting with the file system

#defining the root directory path
root_dir = r"C:\Users\jwboy\Uni work\project\enron_mail_20150507\maildir"

#checking if the paths exist
print("exists:", os.path.exists(root_dir))
print("isdir:", os.path.isdir(root_dir))

#print the first 10 entries found in the directory
subfolders = os.listdir(root_dir)[:10]
print("first 10 entries:", subfolders)

exists: True
isdir: True
first 10 entries: ['allen-p', 'arnold-j', 'arora-h', 'badeer-r', 'bailey-s', 'bass-e', 'baughman-d', 'beck-s', 'benson-r', 'blair-l']


In [3]:
#confirming the structure of the dataset and the emails are reachable

#building a path to a specific user's email folder
sample_folder = os.path.join(root_dir, "allen-p", "all_documents")
#checking the folder exists
print("sample folder exists:", os.path.exists(sample_folder))
#checking that the path isn't a file
print("isdir:", os.path.isdir(sample_folder))

#list the first 10 files in the folder
files = os.listdir(sample_folder)[:10]
print("first 10 files:", files)

#building the full path to the first file in the list
sample_file = os.path.join(sample_folder, files[0])
print("sample file path:", sample_file)

sample folder exists: True
isdir: True
first 10 files: ['1.', '10.', '100.', '101.', '102.', '103.', '104.', '105.', '106.', '107.']
sample file path: C:\Users\jwboy\Uni work\project\enron_mail_20150507\maildir\allen-p\all_documents\1.


In [4]:
#creating a more safe path for accessing the email files
sample_folder = os.path.join(root_dir, "allen-p", "all_documents")
name = os.listdir(sample_folder)[0]

#two paths are built one for normal access and the other for
#long path versions
normal_path = os.path.join(sample_folder, name)
verbatim_path = r"\\?\\" + normal_path

#checking that the path is robust
print("name:", repr(name))
print("verbatim exists:", os.path.exists(verbatim_path))
print("verbatim isfile:", os.path.isfile(verbatim_path))
print("verbatim path:", verbatim_path)

name: '1.'
verbatim exists: True
verbatim isfile: True
verbatim path: \\?\\C:\Users\jwboy\Uni work\project\enron_mail_20150507\maildir\allen-p\all_documents\1.


In [5]:
#open and insepct the first 1k characters in a file
#checking the content of the emails
with open(verbatim_path, "r", encoding="latin-1", errors="ignore") as f:
    text = f.read(1000)

print(text)

Message-ID: <29790972.1075855665306.JavaMail.evans@thyme>
Date: Wed, 13 Dec 2000 18:41:00 -0800 (PST)
From: 1.11913372.-2@multexinvestornetwork.com
To: pallen@enron.com
Subject: December 14, 2000 - Bear Stearns' predictions for telecom in Latin
 America
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Multex Investor <1.11913372.-2@multexinvestornetwork.com>
X-To: <pallen@enron.com>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Dec2000\Notes Folders\All documents
X-Origin: Allen-P
X-FileName: pallen.nsf

In today's Daily Update you'll find free reports on
America Online (AOL), Divine Interventures (DVIN),
and 3M (MMM); reports on the broadband space, Latin
American telecom, and more.

For free research, editor's picks, and more come to the Daily Investor:
http://www.multexinvestor.com/AF004627/magazinecover.asp?promo=unl&d=20001214#
investor

***************************************************************
You are receiving this mail because

In [6]:
#parse the file as an email message
import email

sample_folder = os.path.join(root_dir, "allen-p", "all_documents")
name = os.listdir(sample_folder)[0]

normal_path = os.path.join(sample_folder, name)
path = r"\\?\\" + normal_path

with open(path, "r", encoding="latin-1", errors="ignore") as f:
    msg = email.message_from_file(f)

#getting headers
print("From   :", msg.get("From"))
print("To     :", msg.get("To"))
print("Subject:", msg.get("Subject"))
print("Date   :", msg.get("Date"))
print()
print("Payload preview:")
print(str(msg.get_payload())[:1000])

From   : 1.11913372.-2@multexinvestornetwork.com
To     : pallen@enron.com
Subject: December 14, 2000 - Bear Stearns' predictions for telecom in Latin
 America
Date   : Wed, 13 Dec 2000 18:41:00 -0800 (PST)

Payload preview:
In today's Daily Update you'll find free reports on
America Online (AOL), Divine Interventures (DVIN),
and 3M (MMM); reports on the broadband space, Latin
American telecom, and more.

For free research, editor's picks, and more come to the Daily Investor:
http://www.multexinvestor.com/AF004627/magazinecover.asp?promo=unl&d=20001214#
investor

***************************************************************
You are receiving this mail because you have registered for
Multex Investor. To unsubscribe, see bottom of this message.
***************************************************************

======================== Sponsored by =========================
Would you own just the energy stocks in the S&P 500?
Select Sector SPDRs divides the S&P 500 into nine sector index 

In [7]:
#from that one parsed email create a dataframe row
import pandas as pd

record = {
    "from": msg.get("From"),
    "to": msg.get("To"),
    "subject": msg.get("Subject"),
    "date": msg.get("Date"),
    "body": str(msg.get_payload()),
    "file_path": normal_path
}

print(record)
#printing the new df row
df_test = pd.DataFrame([record])
print(df_test.head())
print(df_test.columns.tolist())

{'from': '1.11913372.-2@multexinvestornetwork.com', 'to': 'pallen@enron.com', 'subject': "December 14, 2000 - Bear Stearns' predictions for telecom in Latin\n America", 'date': 'Wed, 13 Dec 2000 18:41:00 -0800 (PST)', 'body': 'In today\'s Daily Update you\'ll find free reports on\nAmerica Online (AOL), Divine Interventures (DVIN),\nand 3M (MMM); reports on the broadband space, Latin\nAmerican telecom, and more.\n\nFor free research, editor\'s picks, and more come to the Daily Investor:\nhttp://www.multexinvestor.com/AF004627/magazinecover.asp?promo=unl&d=20001214#\ninvestor\n\n***************************************************************\nYou are receiving this mail because you have registered for\nMultex Investor. To unsubscribe, see bottom of this message.\n***************************************************************\n\n======================== Sponsored by =========================\nWould you own just the energy stocks in the S&P 500?\nSelect Sector SPDRs divides the S&P 500 in

In [ ]:
#parse a small batch of emails
data = []
count = 0
#loop limit of 10
limit = 10
#small batch parse over the dataset using os.walk
#checking that the single-email logic also works when
#repeated across multiple files
for root, dirs, files in os.walk(root_dir):
    for name in files:
        normal_path = os.path.join(root, name)
        path = r"\\?\\" + normal_path

        try:
            with open(path, "r", encoding="latin-1", errors="ignore") as f:
                msg = email.message_from_file(f)

            record = {
                "from": msg.get("From"),
                "to": msg.get("To"),
                "subject": msg.get("Subject"),
                "date": msg.get("Date"),
                "body": str(msg.get_payload()),
                "file_path": normal_path
            }

            data.append(record)
            count += 1

            if count >= limit:
                break

        except Exception as e:
            print("error:", normal_path, repr(e))

    if count >= limit:
        break

df_small = pd.DataFrame(data)
#printing the results of the small scale pipeline
print("rows:", len(df_small))
print("columns:", df_small.columns.tolist())
print(df_small.head())

rows: 10
columns: ['from', 'to', 'subject', 'date', 'body', 'file_path']
                                      from                        to  \
0  1.11913372.-2@multexinvestornetwork.com          pallen@enron.com   
1              messenger@ecm.bloomberg.com                      None   
2                  phillip.allen@enron.com     keith.holst@enron.com   
3                  phillip.allen@enron.com     keith.holst@enron.com   
4                  phillip.allen@enron.com  david.delainey@enron.com   

                                             subject  \
0  December 14, 2000 - Bear Stearns' predictions ...   
1                       Bloomberg Power Lines Report   
2        Consolidated positions: Issues & To Do list   
3        Consolidated positions: Issues & To Do list   
4                                                      

                                    date  \
0  Wed, 13 Dec 2000 18:41:00 -0800 (PST)   
1  Wed, 13 Dec 2000 08:35:00 -0800 (PST)   
2   Mon, 9 Oct 2000 07:16

In [9]:
#cleaning the email body text
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
#applying the cleaning to the text
df_small["clean_body"] = df_small["body"].apply(clean_text)

print(df_small[["body", "clean_body"]].head(3))

                                                body  \
0  In today's Daily Update you'll find free repor...   
1  Here is today's copy of Bloomberg Power Lines....   
2  ---------------------- Forwarded by Phillip K ...   

                                          clean_body  
0  in todays daily update youll find free reports...  
1  here is todays copy of bloomberg power lines a...  
2  forwarded by phillip k allenhouect on pm richa...  


From here both extraction and cleaning has been tested on
a small scale sample, now it can be scaled up to the desired
amount needed for the dataset

In [10]:
#extracting a larger sample of rows to be used

data = []
count = 0
#10000 as this reduces loading time. In the prep file the dataset
#will have 1000 random rows selected for model implementation
limit = 10000

# Walk through all files under root_dir
for root, dirs, files in os.walk(root_dir):
    for name in files:
        normal_path = os.path.join(root, name)
        path = r"\\?\\" + normal_path
        # Add Windows long path prefix to handle paths with lots of characters

        try:
            # Use latin-1 to prevent decode errors
            with open(path, "r", encoding="latin-1", errors="ignore") as f:
                msg = email.message_from_file(f)

            #What to parse from the email
            record = {
                "from": msg.get("From"),
                "to": msg.get("To"),
                "subject": msg.get("Subject"),
                "date": msg.get("Date"),
                "body": str(msg.get_payload()),
                "file_path": normal_path
            }

            data.append(record)
            count += 1

            # Stop after reaching limit
            if count >= limit:
                break

        except Exception:
            continue

    if count >= limit:
        break

df_sample = pd.DataFrame(data)
#print the extracted data, to check it works
print("rows:", len(df_sample))
print(df_sample.head())

rows: 10000
                                      from                        to  \
0  1.11913372.-2@multexinvestornetwork.com          pallen@enron.com   
1              messenger@ecm.bloomberg.com                      None   
2                  phillip.allen@enron.com     keith.holst@enron.com   
3                  phillip.allen@enron.com     keith.holst@enron.com   
4                  phillip.allen@enron.com  david.delainey@enron.com   

                                             subject  \
0  December 14, 2000 - Bear Stearns' predictions ...   
1                       Bloomberg Power Lines Report   
2        Consolidated positions: Issues & To Do list   
3        Consolidated positions: Issues & To Do list   
4                                                      

                                    date  \
0  Wed, 13 Dec 2000 18:41:00 -0800 (PST)   
1  Wed, 13 Dec 2000 08:35:00 -0800 (PST)   
2   Mon, 9 Oct 2000 07:16:00 -0700 (PDT)   
3   Mon, 9 Oct 2000 07:00:00 -0700 (PDT)  

In [11]:
#cleans that larger sample using the same function as before
df_sample["clean_body"] = df_sample["body"].apply(clean_text)

print(df_sample[["subject", "clean_body"]].head(5))
print(df_sample.shape)

                                             subject  \
0  December 14, 2000 - Bear Stearns' predictions ...   
1                       Bloomberg Power Lines Report   
2        Consolidated positions: Issues & To Do list   
3        Consolidated positions: Issues & To Do list   
4                                                      

                                          clean_body  
0  in todays daily update youll find free reports...  
1  here is todays copy of bloomberg power lines a...  
2  forwarded by phillip k allenhouect on pm richa...  
3  forwarded by phillip k allenhouect on pm richa...  
4  dave here are the names of the west desk membe...  
(10000, 7)


Now that the emails have been extracted, the data can be structured into a useable format, dropping and merging unnecessary columns.

In [12]:
#checking the outcome by printing the top 5 rows
print(df_sample.head())
print(df_sample.columns.tolist())

                                      from                        to  \
0  1.11913372.-2@multexinvestornetwork.com          pallen@enron.com   
1              messenger@ecm.bloomberg.com                      None   
2                  phillip.allen@enron.com     keith.holst@enron.com   
3                  phillip.allen@enron.com     keith.holst@enron.com   
4                  phillip.allen@enron.com  david.delainey@enron.com   

                                             subject  \
0  December 14, 2000 - Bear Stearns' predictions ...   
1                       Bloomberg Power Lines Report   
2        Consolidated positions: Issues & To Do list   
3        Consolidated positions: Issues & To Do list   
4                                                      

                                    date  \
0  Wed, 13 Dec 2000 18:41:00 -0800 (PST)   
1  Wed, 13 Dec 2000 08:35:00 -0800 (PST)   
2   Mon, 9 Oct 2000 07:16:00 -0700 (PDT)   
3   Mon, 9 Oct 2000 07:00:00 -0700 (PDT)   
4   Thu, 5

In [13]:
#saving only what rows are needed

# Keep only the columns needed
df_sample = df_sample[["subject", "clean_body"]]

# Adding label column
df_sample["label"] = "benign"

df_sample["text"] = df_sample["subject"].fillna("") + " " + df_sample["clean_body"].fillna("")
print(df_sample[["text"]].head())

df = df_sample[["label", "subject", "clean_body", "text"]]
df = df[["label", "text"]]
df = df.drop_duplicates()

print(df.head())
print(df.columns)

                                                text
0  December 14, 2000 - Bear Stearns' predictions ...
1  Bloomberg Power Lines Report here is todays co...
2  Consolidated positions: Issues & To Do list fo...
3  Consolidated positions: Issues & To Do list fo...
4   dave here are the names of the west desk memb...
    label                                               text
0  benign  December 14, 2000 - Bear Stearns' predictions ...
1  benign  Bloomberg Power Lines Report here is todays co...
2  benign  Consolidated positions: Issues & To Do list fo...
4  benign   dave here are the names of the west desk memb...
5  benign  Re: 2001 Margin Plan paula million is fine phi...
Index(['label', 'text'], dtype='object')


In [14]:
#Code to select X number of lines from the Enron dataset

# Number of rows needed
n = 1500  # change this

# Random sample
random_df = df.sample(n=n, random_state=42)

# Save to new CSV
random_df.to_csv("1_Enron_ml_ready.csv", index=False)

print(f"Saved {n} random rows to 1_Enron_ml_ready.csv")

Saved 1500 random rows to 1_Enron_ml_ready.csv
